In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_project.silver.users (
    user_id BIGINT,
    name STRING,
    changing_value INT,
    date_from TIMESTAMP,
    date_to TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = f"databricks_project.silver.users"
dim_table = DeltaTable.forName(spark, table_name)

df = spark.sql('''
    SELECT user_id, name, changing_value, ingested_at, 
        ingested_at as date_from,
        null as date_to
    FROM databricks_project.bronze.users
''')

merge_condition = """
    target.user_id = source.user_id 
    AND 
    (target.name != source.name
        OR target.changing_value != source.changing_value)
"""

(dim_table.alias("target")
    .merge(df.alias("source"), merge_condition)
    .whenMatchedUpdate(set={
        "date_to": F.col("source.date_from")
    })
    .whenNotMatchedInsertAll()
    .execute()
)